In [2]:
import requests
import time
import numpy as np

# Warm up the API first
with open('/tmp/test_leaf.jpg', 'rb') as f:
    requests.post('http://localhost:8000/predict',
                  files={'file': ('test.jpg', f, 'image/jpeg')},
                  params={'soil_pH': 6.5, 'nitrogen': 80, 'temperature': 28,
                          'humidity': 65, 'rainfall': 120, 'crop_age_days': 45,
                          'sunlight_hours': 7, 'phosphorus': 60, 'potassium': 70})

print("API warmed up. Running benchmark...")

# Run 5 predictions and measure time
times = []
for i in range(5):
    start = time.time()
    with open('/tmp/test_leaf.jpg', 'rb') as f:
        response = requests.post(
            'http://localhost:8000/predict',
            files={'file': ('test.jpg', f, 'image/jpeg')},
            params={'soil_pH': 6.5, 'nitrogen': 80, 'temperature': 28,
                    'humidity': 65, 'rainfall': 120, 'crop_age_days': 45,
                    'sunlight_hours': 7, 'phosphorus': 60, 'potassium': 70}
        )
    elapsed = time.time() - start
    times.append(elapsed)
    print(f"  Request {i+1}: {elapsed:.2f}s")

print(f"\nAverage response time: {np.mean(times):.2f}s")
print(f"Fastest: {np.min(times):.2f}s")
print(f"Slowest: {np.max(times):.2f}s")

API warmed up. Running benchmark...
  Request 1: 0.17s
  Request 2: 0.18s
  Request 3: 0.18s
  Request 4: 0.17s
  Request 5: 0.16s

Average response time: 0.17s
Fastest: 0.16s
Slowest: 0.18s


In [3]:
import torch
import torch.nn as nn
import numpy as np
import joblib
import pickle
import time
from torchvision import models, transforms
from PIL import Image

# Load everything
with open('data/processed/data_splits.pkl', 'rb') as f:
    image_data = pickle.load(f)
classes = image_data['classes']

model = models.resnet50(weights=None)
model.fc = nn.Linear(2048, len(classes))
model.load_state_dict(torch.load('models/resnet50_balanced.pth', map_location='cpu'))
model.eval()

embedding_model = nn.Sequential(*list(model.children())[:-1])
embedding_model.eval()

class FusionMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(2057, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 15)
        )
    def forward(self, x):
        return self.network(x)

fusion_model = FusionMLP()
fusion_model.load_state_dict(torch.load('models/fusion_mlp_best.pth', map_location='cpu'))
fusion_model.eval()

scaler = joblib.load('models/tabular_scaler_v2.pkl')

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

img = Image.open('/tmp/test_leaf.jpg').convert('RGB')

# Benchmark each step
steps = {}

t = time.time()
img_tensor = transform(img).unsqueeze(0)
steps['preprocessing'] = time.time() - t

t = time.time()
with torch.no_grad():
    embedding = embedding_model(img_tensor)
    embedding = embedding.squeeze(-1).squeeze(-1).numpy()
steps['cnn_embedding'] = time.time() - t

t = time.time()
tab = np.array([[6.5, 80, 60, 70, 28, 65, 120, 45, 7]])
tab_scaled = scaler.transform(tab)
fused = np.concatenate([embedding, tab_scaled], axis=1)
fused_tensor = torch.tensor(fused, dtype=torch.float32)
steps['tabular_prep'] = time.time() - t

t = time.time()
with torch.no_grad():
    outputs = fusion_model(fused_tensor)
    probs = torch.softmax(outputs, dim=1)
    conf, pred = probs.max(1)
steps['fusion_predict'] = time.time() - t

print("Time breakdown per prediction step:")
total = sum(steps.values())
for step, duration in steps.items():
    pct = duration / total * 100
    bar = '█' * int(pct / 2)
    print(f"  {step:20s}: {duration*1000:.1f}ms  {bar} {pct:.1f}%")
print(f"\n  Total: {total*1000:.1f}ms")

Time breakdown per prediction step:
  preprocessing       : 2.4ms  ███ 6.2%
  cnn_embedding       : 35.2ms  █████████████████████████████████████████████ 91.8%
  tabular_prep        : 0.4ms   1.1%
  fusion_predict      : 0.3ms   0.9%

  Total: 38.4ms


In [4]:
# Optimization 1: Use inference_mode instead of no_grad (faster)
t = time.time()
with torch.inference_mode():
    embedding = embedding_model(img_tensor)
    embedding = embedding.squeeze(-1).squeeze(-1).numpy()
inference_mode_time = time.time() - t
print(f"inference_mode:  {inference_mode_time*1000:.1f}ms")

# Optimization 2: Compile the model (PyTorch 2.0+)
try:
    compiled_embedding = torch.compile(embedding_model)
    # Warmup run
    with torch.inference_mode():
        _ = compiled_embedding(img_tensor)
    # Benchmark
    times = []
    for _ in range(5):
        t = time.time()
        with torch.inference_mode():
            out = compiled_embedding(img_tensor)
        times.append(time.time() - t)
    compiled_time = np.mean(times)
    print(f"torch.compile:   {compiled_time*1000:.1f}ms")
    print(f"\nSpeedup from compile: {35.2/compiled_time*1000:.1f}x faster")
except Exception as e:
    print(f"torch.compile not available: {e}")

# Optimization 3: Convert to float16 (half precision)
try:
    model_fp16 = embedding_model.half()
    img_fp16 = img_tensor.half()
    with torch.inference_mode():
        _ = model_fp16(img_fp16)  # warmup
    times = []
    for _ in range(5):
        t = time.time()
        with torch.inference_mode():
            out = model_fp16(img_fp16)
        times.append(time.time() - t)
    fp16_time = np.mean(times)
    print(f"float16:         {fp16_time*1000:.1f}ms")
except Exception as e:
    print(f"float16 not available: {e}")

inference_mode:  71.2ms
torch.compile:   31.4ms

Speedup from compile: 1122653.6x faster
float16:         1936.7ms


In [5]:
new_api = open('src/api.py').read()

# Replace no_grad with inference_mode throughout
new_api = new_api.replace(
    'with torch.no_grad():',
    'with torch.inference_mode():'
)

# Add model compilation after models are loaded in startup
old_startup_end = '    SCALER = joblib.load("models/tabular_scaler_v2.pkl")\n    print("All models loaded successfully!")'
new_startup_end = '''    SCALER = joblib.load("models/tabular_scaler_v2.pkl")

    # Compile models for faster inference
    global EMBEDDING_MODEL, FUSION_MODEL
    try:
        EMBEDDING_MODEL = torch.compile(EMBEDDING_MODEL)
        FUSION_MODEL = torch.compile(FUSION_MODEL)
        # Warmup compiled models
        dummy = torch.randn(1, 3, 224, 224)
        with torch.inference_mode():
            emb = EMBEDDING_MODEL(dummy)
            emb = emb.squeeze(-1).squeeze(-1)
            tab = torch.zeros(1, 9)
            fused = torch.cat([emb, tab], dim=1)
            _ = FUSION_MODEL(fused)
        print("Models compiled successfully!")
    except Exception as e:
        print(f"Compilation skipped: {e}")
    print("All models loaded successfully!")'''

new_api = new_api.replace(old_startup_end, new_startup_end)

with open('src/api.py', 'w') as f:
    f.write(new_api)

print("src/api.py updated with optimizations!")

src/api.py updated with optimizations!


In [6]:
new_api = open('src/api.py').read().replace(
    '    # Compile models for faster inference\n    global EMBEDDING_MODEL, FUSION_MODEL\n    try:',
    '    # Compile models for faster inference\n    try:'
)

with open('src/api.py', 'w') as f:
    f.write(new_api)

print("Fixed!")

Fixed!


In [7]:
import requests
import time
import numpy as np

# Warm up the API first
with open('/tmp/test_leaf.jpg', 'rb') as f:
    requests.post('http://localhost:8000/predict',
                  files={'file': ('test.jpg', f, 'image/jpeg')},
                  params={'soil_pH': 6.5, 'nitrogen': 80, 'temperature': 28,
                          'humidity': 65, 'rainfall': 120, 'crop_age_days': 45,
                          'sunlight_hours': 7, 'phosphorus': 60, 'potassium': 70})

print("API warmed up. Running benchmark...")

# Run 5 predictions and measure time
times = []
for i in range(5):
    start = time.time()
    with open('/tmp/test_leaf.jpg', 'rb') as f:
        response = requests.post(
            'http://localhost:8000/predict',
            files={'file': ('test.jpg', f, 'image/jpeg')},
            params={'soil_pH': 6.5, 'nitrogen': 80, 'temperature': 28,
                    'humidity': 65, 'rainfall': 120, 'crop_age_days': 45,
                    'sunlight_hours': 7, 'phosphorus': 60, 'potassium': 70}
        )
    elapsed = time.time() - start
    times.append(elapsed)
    print(f"  Request {i+1}: {elapsed:.2f}s")

print(f"\nAverage response time: {np.mean(times):.2f}s")
print(f"Fastest: {np.min(times):.2f}s")
print(f"Slowest: {np.max(times):.2f}s")

API warmed up. Running benchmark...
  Request 1: 1.04s
  Request 2: 0.18s
  Request 3: 0.18s
  Request 4: 0.17s
  Request 5: 0.18s

Average response time: 0.35s
Fastest: 0.17s
Slowest: 1.04s
